# Shorts/Reels Generation — Full Highlight Pipeline

```
Phase 1  ->  Geometric Filter (pairwise, improved weights + NMS)
Phase 2  ->  Re-ranker leger sur embeddings ImageBind
Phase 3  ->  Assemblage du short (cuts + transitions)
```

**Input** : embeddings ImageBind deja calcules (`_unified_final.npz`, `_embeddings_final.npz`)  
**Output** : fichier video short `.mp4` pret pour publication

---

In [ ]:
# CELL 1: Mount Drive, Paths & Config
import sys, os
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT   = '/content/drive/MyDrive/PFA'
IMAGEBIND_ROOT = '/content/drive/MyDrive/PFA/ImageBind'
LIB_ROOT       = '/content/drive/MyDrive/PFA/ImageBind/ImageBind'

for p in [LIB_ROOT, IMAGEBIND_ROOT, PROJECT_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

INPUT_DIR  = os.path.join(PROJECT_ROOT, 'data')
OUTPUT_DIR = os.path.join(IMAGEBIND_ROOT, 'results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_VIDEO = 'Obama_Yes_we_can.mp4'   # <- changer ici
VIDEO_PATH   = os.path.join(INPUT_DIR, TARGET_VIDEO)
VIDEO_STEM   = os.path.splitext(TARGET_VIDEO)[0]

CFG = dict(
    WINDOW            = 5,
    TOP_N_GEO         = 20,
    NMS_MIN_GAP_S     = 10.0,
    NMS_TOP_K         = 10,
    W_COHERENCE       = 0.50,
    W_NOVELTY         = 0.30,
    W_SALIENCY        = 0.20,
    RERANKER_MODE     = 'zero_shot',
    RERANKER_TOP_K    = 5,
    TARGET_DURATION_S = 60,
    FADE_DURATION_S   = 0.5,
    ADD_CAPTIONS      = True,
    OUTPUT_RESOLUTION = '1080x1920',
    OUTPUT_FPS        = 30,
)

print('Config chargee')
for k, v in CFG.items():
    print(f'  {k:<25} = {v}')

In [ ]:
# CELL 2: Dependencies
!pip install -q moviepy==1.0.3 librosa

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import json, librosa, subprocess, shutil
from pathlib import Path

print('Dependencies OK')

---
## PHASE 1 — Geometric Filter (ameliore)


In [ ]:
# CELL 3: Load Stage 4 outputs
UNIFIED_PATH = os.path.join(OUTPUT_DIR, VIDEO_STEM + '_unified_final.npz')
EMB_PATH     = os.path.join(OUTPUT_DIR, VIDEO_STEM + '_embeddings_final.npz')

npz     = np.load(UNIFIED_PATH, allow_pickle=True)
unified = torch.tensor(npz['unified'])
times   = npz['times']
texts   = npz['raw_text']
trust   = torch.tensor(npz['text_trust'])

emb_npz = np.load(EMB_PATH, allow_pickle=True)
V = F.normalize(torch.tensor(emb_npz['vision']), dim=-1)
A = F.normalize(torch.tensor(emb_npz['audio']),  dim=-1)
T = F.normalize(torch.tensor(emb_npz['text']),   dim=-1)

N = len(times)
print(f'{N} segments charges')
print(f'unified : {unified.shape}')
print(f'V/A/T   : {V.shape}')
print(f'Duree video totale : {times[-1][1]:.1f}s')

In [ ]:
# CELL 4: Signal 4 — Audio Energy (RMS per segment)
def extract_audio_rms(video_path, times_arr, sr=22050):
    audio_tmp = '/tmp/audio_tmp.wav'
    cmd = ['ffmpeg', '-y', '-i', video_path,
           '-ac', '1', '-ar', str(sr), audio_tmp, '-loglevel', 'error']
    subprocess.run(cmd, check=True)
    y, _ = librosa.load(audio_tmp, sr=sr, mono=True)
    rms_list = []
    for (start_s, end_s) in times_arr:
        s = int(start_s * sr)
        e = int(end_s   * sr)
        chunk = y[s:e]
        rms_list.append(float(np.sqrt(np.mean(chunk**2))) if len(chunk) > 0 else 0.0)
    rms = torch.tensor(rms_list)
    return (rms - rms.min()) / (rms.max() - rms.min() + 1e-8)

audio_energy = extract_audio_rms(VIDEO_PATH, times)
print(f'Audio RMS: mean={audio_energy.mean():.3f}  std={audio_energy.std():.3f}')

In [ ]:
# CELL 5: Compute 4 geometric signals (pairwise)
WINDOW = CFG['WINDOW']

def compute_pairwise_coherence(V, A, T, trust, window=5):
    N = V.shape[0]
    pair_scores = torch.zeros(N - 1)
    def seg_coh(idx):
        w = trust[idx]
        va = (V[idx] * A[idx]).sum()
        vt = (V[idx] * T[idx]).sum() * w
        at = (A[idx] * T[idx]).sum() * w
        return (va + vt + at) / (1.0 + w + w + 1e-8)
    for i in range(N - 1):
        j  = i + 1
        lo = max(0, i - window)
        hi = min(N, j + window + 1)
        ctx = [k for k in range(lo, hi) if k != i and k != j]
        ci = seg_coh(i)
        cj = seg_coh(j)
        cc = torch.stack([seg_coh(k) for k in ctx]).mean() if ctx else torch.tensor(0.0)
        pair_scores[i] = ((ci + cj) / 2.0) - cc
    return pair_scores

def compute_pairwise_novelty(unified, window=5):
    N = len(unified)
    pair_scores = torch.zeros(N - 1)
    for i in range(N - 1):
        j = i + 1
        lc = unified[max(0, i - window):i]
        rc = unified[j + 1:min(N, j + window + 1)]
        ci = F.normalize((lc.mean(dim=0) if len(lc) > 0 else unified[i]).unsqueeze(0), dim=-1).squeeze(0)
        cj = F.normalize((rc.mean(dim=0) if len(rc) > 0 else unified[j]).unsqueeze(0), dim=-1).squeeze(0)
        si = F.normalize(unified[i].unsqueeze(0), dim=-1).squeeze(0)
        sj = F.normalize(unified[j].unsqueeze(0), dim=-1).squeeze(0)
        pair_scores[i] = 0.6 * (1.0 - (si * sj).sum()) + 0.4 * (1.0 - (ci * cj).sum())
    return pair_scores

def compute_pairwise_saliency(unified, window=5):
    N = len(unified)
    gm = F.normalize(unified.mean(dim=0).unsqueeze(0), dim=-1).squeeze(0)
    pair_scores = torch.zeros(N - 1)
    for i in range(N - 1):
        j  = i + 1
        lo = max(0, i - window); hi = min(N, j + window + 1)
        ctx = [k for k in range(lo, hi) if k != i and k != j]
        pm  = F.normalize(((unified[i] + unified[j]) / 2.0).unsqueeze(0), dim=-1).squeeze(0)
        gd  = 1.0 - (pm * gm).sum()
        if ctx:
            lm = F.normalize(unified[ctx].mean(dim=0).unsqueeze(0), dim=-1).squeeze(0)
            ld = 1.0 - (pm * lm).sum()
        else:
            ld = gd
        pair_scores[i] = 0.5 * gd + 0.5 * ld
    return pair_scores

def minmax_norm(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

print('Computing signals...')
raw_coh = compute_pairwise_coherence(V, A, T, trust, WINDOW)
raw_nov = compute_pairwise_novelty(unified, WINDOW)
raw_sal = compute_pairwise_saliency(unified, WINDOW)
raw_aud = (audio_energy[:-1] + audio_energy[1:]) / 2.0

coh_n = minmax_norm(raw_coh)
nov_n = minmax_norm(raw_nov)
sal_n = minmax_norm(raw_sal)
aud_n = minmax_norm(raw_aud)

print('Signals (mean +/- std):')
for name, s in [('Coherence', coh_n), ('Novelty', nov_n), ('Saliency', sal_n), ('Audio', aud_n)]:
    print(f'  {name:<12}: {s.mean():.3f} +/- {s.std():.3f}')

In [ ]:
# CELL 6: Optional grid search for weights (requires KNOWN_HIGHLIGHTS)
# Set KNOWN_HIGHLIGHTS = [(start_s, end_s), ...] if you have manual labels
KNOWN_HIGHLIGHTS = []  # e.g. [(12.0, 22.0), (55.0, 65.0)]

pair_times = np.array([[times[p][0], times[p+1][1]] for p in range(N - 1)])
pair_texts = np.array([str(texts[p]) + ' | ' + str(texts[p+1]) for p in range(N - 1)])

def pair_hits(pair_idx, known, iou_thresh=0.3):
    ps, pe = pair_times[pair_idx]
    for hs, he in known:
        inter = max(0, min(pe, he) - max(ps, hs))
        union = max(pe, he) - min(ps, hs)
        if union > 0 and (inter / union) >= iou_thresh:
            return 1
    return 0

if KNOWN_HIGHLIGHTS:
    labels = torch.tensor([pair_hits(i, KNOWN_HIGHLIGHTS) for i in range(N - 1)], dtype=torch.float)
    print(f'Labels: {labels.sum():.0f} positives / {len(labels)} pairs')
    best_ap, best_w = 0.0, None
    for wc in np.arange(0.3, 0.7, 0.1):
        for wn in np.arange(0.1, 0.5, 0.1):
            ws = round(1.0 - wc - wn, 2)
            if ws < 0.05: continue
            sc = wc * coh_n + wn * nov_n + ws * sal_n
            hits = labels[sc.argsort(descending=True)[:20]].sum().item()
            ap = hits / 20.0
            if ap > best_ap:
                best_ap, best_w = ap, (float(wc), float(wn), float(ws))
    print(f'Best weights: Coh={best_w[0]:.2f} Nov={best_w[1]:.2f} Sal={best_w[2]:.2f}  AP@20={best_ap:.2f}')
    CFG['W_COHERENCE'] = best_w[0]
    CFG['W_NOVELTY']   = best_w[1]
    CFG['W_SALIENCY']  = best_w[2]
else:
    print('No labels -> using default weights (set KNOWN_HIGHLIGHTS to enable grid search)')

W_AUD = 0.15
tot   = CFG['W_COHERENCE'] + CFG['W_NOVELTY'] + CFG['W_SALIENCY']
wc = CFG['W_COHERENCE'] / tot * (1 - W_AUD)
wn = CFG['W_NOVELTY']   / tot * (1 - W_AUD)
ws = CFG['W_SALIENCY']  / tot * (1 - W_AUD)
wa = W_AUD
print(f'Final weights: Coh={wc:.3f} Nov={wn:.3f} Sal={ws:.3f} Aud={wa:.3f}')

In [ ]:
# CELL 7: Combined Geo Score + Temporal NMS
geo_score = wc * coh_n + wn * nov_n + ws * sal_n + wa * aud_n

def temporal_nms(scores, pt, min_gap_s, top_k):
    sorted_idx = scores.argsort(descending=True).tolist()
    kept = []
    for idx in sorted_idx:
        if len(kept) >= top_k: break
        s_new = (pt[idx][0] + pt[idx][1]) / 2.0
        too_close = any(
            (max(0, min(pt[idx][1], pt[k][1]) - max(pt[idx][0], pt[k][0])) > 0)
            or (abs(s_new - (pt[k][0] + pt[k][1]) / 2.0) < min_gap_s)
            for k in kept
        )
        if not too_close:
            kept.append(idx)
    return kept

nms_kept = temporal_nms(geo_score, pair_times, CFG['NMS_MIN_GAP_S'], CFG['NMS_TOP_K'])
print(f'NMS: {len(nms_kept)} highlights kept (gap >= {CFG["NMS_MIN_GAP_S"]}s)')
print()
print(f'{"Rk":<4} {"Segs":>9}  {"Start":>7} {"End":>7}  {"Score":>7}  Text')
print('-' * 85)
for rank, idx in enumerate(nms_kept):
    p = idx
    print(f'#{rank+1:<3} [{p:>3}+{p+1:<3}]  '
          f'{pair_times[p][0]:>6.1f}s {pair_times[p][1]:>6.1f}s  '
          f'{geo_score[p]:>7.3f}  {pair_texts[p][:55]}')

In [ ]:
# CELL 8: Save Phase 1 output
P1_PATH = os.path.join(OUTPUT_DIR, VIDEO_STEM + '_phase1_candidates.npz')
kept_arr = np.array(nms_kept)

np.savez(
    P1_PATH,
    pair_idx  = kept_arr,
    seg_j     = kept_arr + 1,
    geo_score = geo_score[kept_arr].numpy(),
    coherence = coh_n[kept_arr].numpy(),
    novelty   = nov_n[kept_arr].numpy(),
    saliency  = sal_n[kept_arr].numpy(),
    audio_rms = aud_n[kept_arr].numpy(),
    times     = pair_times[kept_arr],
    raw_text  = pair_texts[kept_arr],
)
print(f'Phase 1 saved -> {P1_PATH}')
print(f'{len(nms_kept)} candidates  shape times = {pair_times[kept_arr].shape}')

In [ ]:
# CELL 9: Visualize Phase 1
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
mid_pair = (pair_times[:, 0] + pair_times[:, 1]) / 2

axes[0].plot(mid_pair, coh_n.numpy(), label='Coherence', alpha=0.8)
axes[0].plot(mid_pair, nov_n.numpy(), label='Novelty',   alpha=0.8)
axes[0].plot(mid_pair, sal_n.numpy(), label='Saliency',  alpha=0.8)
axes[0].plot(mid_pair, aud_n.numpy(), label='Audio RMS', alpha=0.8, linestyle='--')
axes[0].set_title('Geometric signals (normalized, pairwise)')
axes[0].legend(loc='upper right')
axes[0].set_ylabel('Score [0-1]')

axes[1].plot(mid_pair, geo_score.numpy(), color='steelblue', label='Geo Score')
for idx in nms_kept:
    axes[1].axvspan(pair_times[idx][0], pair_times[idx][1], alpha=0.3, color='tomato')
axes[1].set_title('Combined geo score + retained highlights (NMS, red)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Score [0-1]')
axes[1].legend()

plt.tight_layout()
viz_path = os.path.join(OUTPUT_DIR, VIDEO_STEM + '_phase1_viz.png')
plt.savefig(viz_path, dpi=120)
plt.show()
print(f'Visualization saved -> {viz_path}')

---
## PHASE 2 — Re-ranker leger sur embeddings ImageBind

**Mode zero_shot** : centralite semantique + diversite + emphase narrative  
**Mode supervised** : MLP [517->256->64->1] si KNOWN_HIGHLIGHTS defini


In [ ]:
# CELL 10: Build re-ranker features [K, 517]
p1 = np.load(P1_PATH, allow_pickle=True)
kept_pairs       = p1['pair_idx']   # [K]
pair_times_cands = p1['times']      # [K, 2]
K = len(kept_pairs)

unified_norm = F.normalize(unified, dim=-1)
pair_emb = []
for p in kept_pairs:
    i, j = int(p), int(p) + 1
    emb = (unified_norm[i] + unified_norm[j]) / 2.0
    pair_emb.append(F.normalize(emb.unsqueeze(0), dim=-1).squeeze(0))
pair_emb = torch.stack(pair_emb)  # [K, 512]

geo_feats = torch.tensor(np.stack([
    p1['coherence'], p1['novelty'], p1['saliency'], p1['audio_rms']
], axis=1), dtype=torch.float)    # [K, 4]

dur = torch.tensor(pair_times_cands[:,1] - pair_times_cands[:,0], dtype=torch.float).unsqueeze(1)
dur = (dur - dur.min()) / (dur.max() - dur.min() + 1e-8)

X = torch.cat([pair_emb, geo_feats, dur], dim=1)  # [K, 517]
print(f'Feature matrix: {X.shape}  (512 emb + 4 geo + 1 dur)')

In [ ]:
# CELL 11: Re-ranker zero-shot
def zero_shot_reranker(pair_emb, geo_feats, pair_times_cands, geo_score_cands):
    K = pair_emb.shape[0]

    # (A) Semantic centrality
    centroid   = F.normalize(pair_emb.mean(dim=0).unsqueeze(0), dim=-1)
    centrality = (pair_emb * centroid).sum(dim=1)
    centrality = (centrality - centrality.min()) / (centrality.max() - centrality.min() + 1e-8)

    # (B) Diversity (inverse max similarity to other candidates)
    sim_mat = pair_emb @ pair_emb.T
    sim_mat.fill_diagonal_(-1)
    diversity = 1.0 - sim_mat.max(dim=1).values
    diversity = (diversity - diversity.min()) / (diversity.max() - diversity.min() + 1e-8)

    # (C) Narrative emphasis (directional change in embedding space)
    time_order = np.argsort(pair_times_cands[:, 0])
    emb_sorted = pair_emb[time_order]
    narrative  = torch.zeros(K)
    for rank, orig in enumerate(time_order):
        prev = rank - 1 if rank > 0     else rank
        nxt  = rank + 1 if rank < K - 1 else rank
        vin  = F.normalize((emb_sorted[rank] - emb_sorted[prev]).unsqueeze(0), dim=-1).squeeze(0)
        vout = F.normalize((emb_sorted[nxt]  - emb_sorted[rank]).unsqueeze(0), dim=-1).squeeze(0)
        narrative[orig] = 1.0 - (vin * vout).sum()
    narrative = (narrative - narrative.min()) / (narrative.max() - narrative.min() + 1e-8)

    final = 0.50 * geo_score_cands + 0.20 * centrality + 0.15 * diversity + 0.15 * narrative
    return final, {'centrality': centrality, 'diversity': diversity, 'narrative': narrative}

geo_score_cands = torch.tensor(p1['geo_score'], dtype=torch.float)
final_score_zs, comps = zero_shot_reranker(pair_emb, geo_feats, pair_times_cands, geo_score_cands)

print('Zero-shot re-ranker results:')
top_zs = final_score_zs.argsort(descending=True)[:CFG['RERANKER_TOP_K']]
print(f'{"Rk":<4} {"Start":>7} {"End":>7}  {"Final":>7} {"Geo":>6} {"Cent":>6} {"Div":>6} {"Narr":>6}  Text')
print('-' * 105)
for rank, k in enumerate(top_zs):
    ki = k.item()
    print(f'#{rank+1:<3} {pair_times_cands[ki][0]:>6.1f}s {pair_times_cands[ki][1]:>6.1f}s  '
          f'{final_score_zs[ki]:>7.3f} {geo_score_cands[ki]:>6.3f} '
          f'{comps["centrality"][ki]:>6.3f} '
          f'{comps["diversity"][ki]:>6.3f} '
          f'{comps["narrative"][ki]:>6.3f}  '
          f'{str(p1["raw_text"][ki])[:50]}')

In [ ]:
# CELL 12: Re-ranker supervised (optional — requires KNOWN_HIGHLIGHTS)
class HighlightMLP(nn.Module):
    def __init__(self, input_dim=517):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.GELU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

if CFG['RERANKER_MODE'] == 'supervised' and KNOWN_HIGHLIGHTS:
    y = torch.tensor([pair_hits(int(p), KNOWN_HIGHLIGHTS) for p in kept_pairs], dtype=torch.float)
    print(f'Supervised labels: {y.sum():.0f} positives / {K} candidates')
    if y.sum() < 2:
        print('Not enough positives - fallback to zero_shot')
        final_score = final_score_zs
    else:
        model = HighlightMLP(input_dim=X.shape[1])
        opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        crit  = nn.BCELoss()
        pos_idx = (y == 1).nonzero(as_tuple=True)[0]
        reps    = max(1, len(y) // (len(pos_idx) + 1))
        X_aug   = torch.cat([X, X[pos_idx].repeat(reps, 1)], dim=0)
        y_aug   = torch.cat([y, torch.ones(len(pos_idx) * reps)], dim=0)
        model.train()
        for epoch in range(300):
            opt.zero_grad()
            loss = crit(model(X_aug), y_aug)
            loss.backward(); opt.step()
            if (epoch + 1) % 50 == 0:
                print(f'  Epoch {epoch+1}/300  loss={loss.item():.4f}')
        model.eval()
        with torch.no_grad():
            final_score = model(X)
        print('Supervised re-ranker trained')
else:
    final_score = final_score_zs
    print('Using zero_shot re-ranker (set RERANKER_MODE="supervised" + KNOWN_HIGHLIGHTS to use MLP)')

In [ ]:
# CELL 13: Save Phase 2 output
final_np  = final_score.detach().numpy()
top_k_idx = np.argsort(final_np)[::-1][:CFG['RERANKER_TOP_K']]

P2_PATH = os.path.join(OUTPUT_DIR, VIDEO_STEM + '_phase2_highlights.npz')
np.savez(
    P2_PATH,
    rank        = np.arange(1, len(top_k_idx) + 1),
    pair_idx    = kept_pairs[top_k_idx],
    final_score = final_np[top_k_idx],
    geo_score   = p1['geo_score'][top_k_idx],
    times       = pair_times_cands[top_k_idx],
    raw_text    = p1['raw_text'][top_k_idx],
)
print(f'Phase 2 saved -> {P2_PATH}')
print(f'{len(top_k_idx)} final highlights')
total_raw = (pair_times_cands[top_k_idx][:,1] - pair_times_cands[top_k_idx][:,0]).sum()
print(f'Total raw duration: {total_raw:.1f}s')

---
## PHASE 3 — Assemblage du Short

1. Decouper les clips (FFmpeg, recadrage portrait 9:16)
2. Captions optionnelles
3. Fondus xfade entre clips
4. Encodage final


In [ ]:
# CELL 14: Extract and resize clips
CLIPS_DIR = os.path.join(OUTPUT_DIR, 'clips_tmp')
os.makedirs(CLIPS_DIR, exist_ok=True)

p2 = np.load(P2_PATH, allow_pickle=True)
hl_times = p2['times']
hl_texts = p2['raw_text']

# Sort chronologically
chron = np.argsort(hl_times[:, 0])
hl_times = hl_times[chron]
hl_texts = hl_texts[chron]

OUT_W, OUT_H = map(int, CFG['OUTPUT_RESOLUTION'].split('x'))

def extract_clip(video_path, start_s, end_s, out_path, w, h, fps):
    dur = end_s - start_s
    vf  = f'crop=ih*{w}/{h}:ih,scale={w}:{h}:flags=lanczos,fps={fps}'
    cmd = ['ffmpeg', '-y', '-ss', str(start_s), '-i', video_path,
           '-t', str(dur), '-vf', vf,
           '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
           '-c:a', 'aac', '-b:a', '128k', '-movflags', '+faststart',
           out_path, '-loglevel', 'error']
    subprocess.run(cmd, check=True)

clip_paths = []
total_dur  = 0.0

for rank, (start_s, end_s) in enumerate(hl_times):
    remaining  = CFG['TARGET_DURATION_S'] - total_dur
    if remaining <= 0:
        print(f'  Target duration reached, skipping clip #{rank+1}')
        continue
    actual_end = min(end_s, start_s + remaining)
    actual_dur = actual_end - start_s
    out_clip   = os.path.join(CLIPS_DIR, f'clip_{rank+1:02d}.mp4')
    extract_clip(VIDEO_PATH, start_s, actual_end, out_clip, OUT_W, OUT_H, CFG['OUTPUT_FPS'])
    clip_paths.append(out_clip)
    total_dur += actual_dur
    print(f'  Clip {rank+1:02d}: [{start_s:.1f}s -> {actual_end:.1f}s]  ({actual_dur:.1f}s)  "{str(hl_texts[rank])[:40]}"')

print(f'{len(clip_paths)} clips extracted - total duration = {total_dur:.1f}s')

In [ ]:
# CELL 15: Add captions (optional)
def add_caption(clip_path, caption, out_path, w, h):
    safe = caption.replace("'", "").replace(':', ' ').replace('%', 'pct')[:80]
    fs   = max(24, w // 32)
    mg   = h // 20
    vf   = (f"drawtext=text='{safe}':fontsize={fs}:fontcolor=white"
            f":bordercolor=black:borderw=3:x=(w-text_w)/2:y=h-text_h-{mg}"
            f":box=1:boxcolor=black@0.5:boxborderw=10")
    cmd = ['ffmpeg', '-y', '-i', clip_path, '-vf', vf,
           '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
           '-c:a', 'copy', out_path, '-loglevel', 'error']
    subprocess.run(cmd, check=True)

if CFG['ADD_CAPTIONS']:
    capped = []
    for rank, (cp, txt) in enumerate(zip(clip_paths, hl_texts)):
        out_cap = cp.replace('.mp4', '_cap.mp4')
        caption_clean = str(txt).split(' | ')[0]
        add_caption(cp, caption_clean, out_cap, OUT_W, OUT_H)
        capped.append(out_cap)
        print(f'  Caption clip {rank+1:02d} done')
    clip_paths = capped
    print('Captions added')
else:
    print('Captions disabled (ADD_CAPTIONS=False)')

---
### PHASE 3 — Option B : Ordre Viral (accroche → développement → climax)

Structure classique des shorts viraux :
- **Clip 1** : accroche — le 2ème meilleur score (surprenant, pas le meilleur pour garder la tension)
- **Clips 2 à N-1** : développement — les clips restants en ordre chronologique
- **Clip final** : climax — le meilleur score absolu

*(La Cell 16 originale reste disponible pour l'ordre chronologique classique)*


In [ ]:
# CELL 15b: Reorder clips - Viral structure (hook -> development -> climax)
# Requires: clip_paths and hl_times already set by Cell 14/15

import numpy as np

def viral_reorder(clip_paths, hl_times, final_score_arr):
    """
    Reorder clips for viral short structure:
      - Position 0 : hook     = 2nd highest score (surprising, not the best)
      - Position 1..N-2 : development = remaining clips in chronological order
      - Position N-1 : climax  = highest score

    Args:
        clip_paths     : list of clip file paths (already in chron order from Cell 14)
        hl_times       : np.array [K, 2] timestamps in chron order
        final_score_arr: np.array [K] scores in chron order (same index as clip_paths)

    Returns:
        ordered_paths  : reordered list of clip paths
        ordered_times  : reordered timestamps
        ordered_labels : list of role labels for reporting
    """
    K = len(clip_paths)

    if K == 1:
        return clip_paths, hl_times, ['climax']

    if K == 2:
        # hook = index 1 (2nd best), climax = index 0 (best)
        score_order = np.argsort(final_score_arr)[::-1]  # desc
        best_idx   = score_order[0]
        hook_idx   = score_order[1]
        ordered_idx    = [hook_idx, best_idx]
        ordered_labels = ['hook', 'climax']
        return [clip_paths[i] for i in ordered_idx], hl_times[ordered_idx], ordered_labels

    # K >= 3
    score_order = np.argsort(final_score_arr)[::-1]  # indices sorted by score desc
    best_idx    = score_order[0]   # climax
    hook_idx    = score_order[1]   # hook (2nd best)

    # Development = all others, sorted chronologically by start time
    dev_indices = [i for i in range(K) if i != best_idx and i != hook_idx]
    dev_indices_chron = sorted(dev_indices, key=lambda i: hl_times[i][0])

    ordered_idx    = [hook_idx] + dev_indices_chron + [best_idx]
    ordered_labels = (['hook']
                      + ['dev'] * len(dev_indices_chron)
                      + ['climax'])

    return [clip_paths[i] for i in ordered_idx], hl_times[ordered_idx], ordered_labels


# Build score array aligned with chron-ordered clip_paths
# hl_times and hl_texts are already chron-sorted (Cell 14)
# We need to map back to final_score (which was scored in p1/p2 order)
# Strategy: re-score by matching timestamps
p2_times  = p2['times']          # [K, 2] in p2 order
p2_scores = p2['final_score']    # [K]    in p2 order

chron_scores = np.zeros(len(clip_paths))
for ci, (cs, ce) in enumerate(hl_times[:len(clip_paths)]):
    # Find matching p2 entry by start time
    diffs = np.abs(p2_times[:, 0] - cs)
    best_match = np.argmin(diffs)
    chron_scores[ci] = p2_scores[best_match]

viral_paths, viral_times, viral_labels = viral_reorder(
    clip_paths, hl_times[:len(clip_paths)], chron_scores
)

print('Viral clip order:')
print(f'{"Pos":<5} {"Role":<12} {"Start":>7} {"End":>7}  {"Score":>7}')
print('-' * 50)
viral_indices = ([list(hl_times[:len(clip_paths)][:,0]).index(t[0])
                  if t[0] in hl_times[:len(clip_paths)][:,0] else i
                  for i, t in enumerate(viral_times)])
for pos, (label, t, sc) in enumerate(zip(viral_labels, viral_times, chron_scores[viral_indices] if len(viral_indices) == len(viral_times) else [0]*len(viral_times))):
    emoji = {'hook': '🎣', 'dev': '📖', 'climax': '🔥'}.get(label, '')
    print(f'{pos+1:<5} {emoji} {label:<10} {t[0]:>6.1f}s {t[1]:>6.1f}s')
print()
print('clip_paths_viral and viral_times ready for Cell 16b (viral assembly)')

In [ ]:
# CELL 16b: Assemble viral short (hook -> development -> climax)
# Cell 16 (chronological) is kept unchanged above.
# Run this cell instead of / in addition to Cell 16.

SHORT_VIRAL_PATH = os.path.join(OUTPUT_DIR, VIDEO_STEM + '_short_viral.mp4')

print(f'Assembling VIRAL short ({len(viral_paths)} clips)...')
print('Order:', ' -> '.join(viral_labels))
print()

assemble_xfade(viral_paths, CFG['FADE_DURATION_S'], SHORT_VIRAL_PATH)

viral_dur = get_duration(SHORT_VIRAL_PATH)
print(f'Viral short assembled -> {SHORT_VIRAL_PATH}')
print(f'Duration  : {viral_dur:.1f}s')
print(f'Resolution: {CFG["OUTPUT_RESOLUTION"]} @ {CFG["OUTPUT_FPS"]}fps')
print()
print('Summary:')
print(f'  _short.mp4        -> chronological order ({len(clip_paths)} clips)')
print(f'  _short_viral.mp4  -> viral order         ({len(viral_paths)} clips)')

In [ ]:
# CELL 16: Assemble clips with xfade transitions
def get_duration(path):
    r = subprocess.run(['ffprobe', '-v', 'error', '-show_entries',
                        'format=duration', '-of', 'csv=p=0', path],
                       capture_output=True, text=True)
    return float(r.stdout.strip())

def assemble_xfade(clip_paths, fade_dur, output_path):
    if len(clip_paths) == 1:
        shutil.copy(clip_paths[0], output_path)
        return
    durations = [get_duration(p) for p in clip_paths]
    inputs = []
    for p in clip_paths:
        inputs += ['-i', p]
    fv, fa = [], []
    pv, pa = '[0:v]', '[0:a]'
    offset = 0.0
    for i in range(1, len(clip_paths)):
        offset += durations[i-1] - fade_dur
        ov = f'[v{i}]' if i < len(clip_paths)-1 else '[vout]'
        oa = f'[a{i}]' if i < len(clip_paths)-1 else '[aout]'
        fv.append(f'{pv}[{i}:v]xfade=transition=fade:duration={fade_dur}:offset={offset:.3f}{ov}')
        fa.append(f'{pa}[{i}:a]acrossfade=d={fade_dur}{oa}')
        pv, pa = ov, oa
    fc = ';'.join(fv + fa)
    cmd = ['ffmpeg', '-y'] + inputs + [
        '-filter_complex', fc,
        '-map', '[vout]', '-map', '[aout]',
        '-c:v', 'libx264', '-preset', 'fast', '-crf', '18',
        '-c:a', 'aac', '-b:a', '192k', '-movflags', '+faststart',
        output_path, '-loglevel', 'error'
    ]
    subprocess.run(cmd, check=True)

SHORT_PATH = os.path.join(OUTPUT_DIR, VIDEO_STEM + '_short.mp4')
print(f'Assembling {len(clip_paths)} clips...')
assemble_xfade(clip_paths, CFG['FADE_DURATION_S'], SHORT_PATH)
final_dur = get_duration(SHORT_PATH)
print(f'Short assembled -> {SHORT_PATH}')
print(f'Final duration  : {final_dur:.1f}s')
print(f'Resolution      : {CFG["OUTPUT_RESOLUTION"]} @ {CFG["OUTPUT_FPS"]}fps')

In [ ]:
# CELL 17: Cleanup & Final Report
shutil.rmtree(CLIPS_DIR, ignore_errors=True)
print('Temp files removed')
print()
print('=' * 65)
print('  PIPELINE REPORT')
print('=' * 65)
print(f'  Source video      : {TARGET_VIDEO}')
print(f'  Segments total    : {N}')
print(f'  Phase 1 candidates: {len(nms_kept)}  (NMS gap >= {CFG["NMS_MIN_GAP_S"]}s)')
print(f'  Phase 2 highlights: {len(top_k_idx)}  (reranker: {CFG["RERANKER_MODE"]})')
print(f'  Output short      : {os.path.basename(SHORT_PATH)}')
print(f'  Duration          : {final_dur:.1f}s / {CFG["TARGET_DURATION_S"]}s target')
print(f'  Resolution        : {CFG["OUTPUT_RESOLUTION"]} portrait 9:16')
print('=' * 65)
print()
print('Highlights (chronological):')
for rank in range(len(clip_paths)):
    s, e = hl_times[rank]
    print(f'  [{rank+1}] {s:>6.1f}s -> {e:>6.1f}s  | {str(hl_texts[rank])[:55]}')